# 作业1：Data Types, Functions and Conditionals

学号：20244080309　姓名：李亦帆

In [1]:
import os, uuid, hashlib, platform, getpass

# ===== 1. 填写学号 =====
STUDENT_ID = 20244080309  # 修改为自己的学号

# ===== 2. 打印 conda 环境 =====
CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV", "base")
print(f"[Conda 环境] {CONDA_ENV}")

# ===== 3. 机器指纹（绑定当前电脑）=====
MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]

# ===== 4. 生成唯一 ID =====
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]      {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")

[Conda 环境] pyda
[学号]      20244080309
[机器指纹] e1e9919215dfdc26
[RUN_ID]   20244080309-e1e9919215dfdc26


## 一、热身：简单函数

In [2]:
def say_hi():
    print("Hello, world!")


say_hi()

Hello, world!


In [3]:
def goat_pad(s):
    print("goat" + s + "goat")


goat_pad("cat")

goatcatgoat


In [4]:
def print_n(s, n):
    for _ in range(n):
        print(s)


print_n("goat", 3)

goat
goat
goat


## 二、欧几里得算法

In [5]:
def gcd(m, n):
    while n != 0:
        m, n = n, m % n
    return m


pairs = [(2018, 2019), (1200, 300), (5040, 60)]
for i, (a, b) in enumerate(pairs, start=1):
    print(f"{i}. gcd({a}, {b}) = {gcd(a, b)}")

1. gcd(2018, 2019) = 1
2. gcd(1200, 300) = 300
3. gcd(5040, 60) = 60


In [6]:
for a, b in [(-18, 24), (18, -24), (-18, -24)]:
    print(f"gcd({a}, {b}) = {gcd(a, b)}")

gcd(-18, 24) = 6
gcd(18, -24) = -6
gcd(-18, -24) = -6


**负数参数**：函数没有做正负处理，结果由取模规则决定——`a % b` 的符号跟随除数 `b`，所以返回值可能带负号，并且随参数顺序改变符号：

- `gcd(-18, 24) = 6`、`gcd(18, -24) = -6`、`gcd(-18, -24) = -6`；
- 返回值的绝对值仍是真正的最大公约数，但负的 GCD 不符合定义（整除与符号无关），也不能直接当作公约数使用，所以这种行为没有实际意义；
- 正确的做法是约定输入为正整数（本题即如此），或在计算前取绝对值。

## 三、近似欧拉数

$$e=\sum_{k=0}^{\infty}\frac{1}{k!}=1+\frac{1}{1!}+\frac{1}{2!}+\frac{1}{6}+\cdots \qquad (1) \qquad\qquad e=\lim_{x\to\infty}\left(1+\frac{1}{x}\right)^{x} \qquad (2)$$

其中 $k! = k\cdot(k-1)\cdots3\cdot2\cdot1$，约定 $0! = 1$。

式(1) 从 $k=0$ 开始计数、函数参数从 $n=1$ 开始计数，所以 n 项对应 $k=0..n-1$：`euler_infinite_sum(0)` 是空和，`euler_infinite_sum(1)` 只取 $1/0! = 1$。

In [7]:
import math


def euler_limit(n):
    return (1 + 1 / n) ** n


print(f"math.exp(1)         = {math.exp(1):.15f}")
for n in (10, 100, 1000, 10000):
    print(f"euler_limit({n:<6}) = {euler_limit(n):.15f}")

math.exp(1)         = 2.718281828459045
euler_limit(10    ) = 2.593742460100002
euler_limit(100   ) = 2.704813829421528
euler_limit(1000  ) = 2.716923932235594
euler_limit(10000 ) = 2.718145926824926


In [8]:
def euler_infinite_sum(n):
    total, term = 0.0, 1.0          # term 依次为 1/0!、1/1!、1/2! ...
    for k in range(n):
        total += term
        term /= (k + 1)
    return total


for k in (0, 1, 4):
    print(f"euler_infinite_sum({k}) = {euler_infinite_sum(k)}")

euler_infinite_sum(0) = 0.0
euler_infinite_sum(1) = 1.0
euler_infinite_sum(4) = 2.6666666666666665


In [9]:
def euler_approx(epsilon):
    n, approx, previous = 0, 0.0, None
    while abs(approx - math.exp(1)) >= epsilon and approx != previous:
        previous = approx
        n += 1
        approx = euler_infinite_sum(n)
    return approx


for epsilon in (1e-2, 1e-6, 1e-10, 1e-15):
    approx = euler_approx(epsilon)
    print(f"epsilon={epsilon:<8g} 近似值={approx:.16f}  误差={abs(approx - math.exp(1)):.3e}")

epsilon=0.01     近似值=2.7083333333333330  误差=9.948e-03
epsilon=1e-06    近似值=2.7182815255731922  误差=3.029e-07
epsilon=1e-10    近似值=2.7182818284467594  误差=1.229e-11
epsilon=1e-15    近似值=2.7182818284590455  误差=4.441e-16


In [10]:
def print_euler_sum_table(n):
    for k in range(1, n + 1):
        print(euler_infinite_sum(k))


print_euler_sum_table(5)

1.0
2.0
2.5
2.6666666666666665
2.708333333333333


`epsilon` 小于 1 ulp（约 4.44e-16）时条件永远不会成立：部分和在浮点下停在 `2.7182818284590455`，与 `math.exp(1) = 2.718281828459045` 相差 1 ulp，所以循环以「部分和不再变化」收尾并返回当前最优值。

## 四、测试整数属性

$n$ 是 2 的幂，当且仅当 $n = 2^p$（$p \ge 0$）。

In [11]:
def is_power_of_2(n):
    if n < 1:
        return False
    while n % 2 == 0:
        n //= 2
    return n == 1


for n in (1, 2, 3, 4, 16, 1023, 1024, 4096):
    print(f"is_power_of_2({n:<5}) = {is_power_of_2(n)}")

is_power_of_2(1    ) = True
is_power_of_2(2    ) = True
is_power_of_2(3    ) = False
is_power_of_2(4    ) = True
is_power_of_2(16   ) = True
is_power_of_2(1023 ) = False
is_power_of_2(1024 ) = True
is_power_of_2(4096 ) = True


In [12]:
def is_power(b, n):
    if b < 1 or n < 1:
        return False
    if b == 1:                      # n % 1 == 0 永远成立，需单独处理
        return n == 1
    while n % b == 0:
        n //= b
    return n == 1


tests = [(2, 1), (2, 1024), (2, 12), (3, 27), (3, 81), (10, 1000), (1, 1), (1, 5)]
for b, n in tests:
    print(f"is_power({b}, {n}) = {is_power(b, n)}")

is_power(2, 1) = True
is_power(2, 1024) = True
is_power(2, 12) = False
is_power(3, 27) = True
is_power(3, 81) = True
is_power(10, 1000) = True
is_power(1, 1) = True
is_power(1, 5) = False
